# 날짜를 날짜답게

> 파이썬 4강 · 데이터 다루기

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [날짜를 날짜답게](https://mioon1402.github.io/timeseriesdata/python/p04-datetime.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales_messy.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 왜 굳이 변환해야 하나

## 2. to_datetime

**4-1. 문자열 → 날짜**

In [ ]:
import pandas as pd
df = pd.read_csv("cafe_sales.csv")

print("변환 전:", df["date"].dtype)

df["date"] = pd.to_datetime(df["date"])

print("변환 후:", df["date"].dtype)
print()
print(df["date"].head(3))

## 3. 형식이 섞여 있을 때

**4-2. 섞인 형식 다루기**

In [ ]:
messy = pd.read_csv("cafe_sales_messy.csv")
print("원본 형식:", messy["날짜"].head(3).tolist())

변환 = pd.to_datetime(messy["날짜"], format="mixed", errors="coerce")

print()
print("변환 성공:", 변환.notna().sum(), "건")
print("변환 실패:", 변환.isna().sum(), "건")
print()
print("실패한 것들:", messy.loc[변환.isna(), "날짜"].head(3).tolist())

**4-3. 한국어 날짜 직접 해석하기**

In [ ]:
# 정규식으로 월·일·연도를 뽑아낸다
부분 = messy["날짜"].str.extract(r"(\d+)월\s*(\d+)일,\s*(\d{4})")
부분.columns = ["month", "day", "year"]

한국식 = pd.to_datetime(부분, errors="coerce")
print("한국식 해석 성공:", 한국식.notna().sum(), "건")

# 두 결과를 합친다 — 먼저 성공한 쪽을 쓰고, 비어 있으면 다른 쪽으로
최종 = 변환.fillna(한국식)
print("최종 변환 성공:", 최종.notna().sum(), "/", len(messy), "건")

최종.head(4)   # 마지막 줄에 값만 두면 표로 보여준다

## 4. dt 접근자

**4-4. 날짜에서 꺼내 쓰기**

In [ ]:
print("연도:  ", df["date"].dt.year.head(3).tolist())
print("월:    ", df["date"].dt.month.head(3).tolist())
print("일:    ", df["date"].dt.day.head(3).tolist())
print("요일번호:", df["date"].dt.dayofweek.head(3).tolist(), "  (0=월요일)")
print("요일이름:", df["date"].dt.day_name().head(3).tolist())
print("주차:  ", df["date"].dt.isocalendar().week.head(3).tolist())
print("분기:  ", df["date"].dt.quarter.head(3).tolist())

## 5. 파생 열 만들기

**4-5. 분석용 열 추가하기**

In [ ]:
df["연"] = df["date"].dt.year
df["월"] = df["date"].dt.month
df["주말"] = df["date"].dt.dayofweek >= 5

# 계절도 만들어보자 — 월을 구간으로 묶는다
def 계절(m):
    if m in (3, 4, 5):   return "봄"
    if m in (6, 7, 8):   return "여름"
    if m in (9, 10, 11): return "가을"
    return "겨울"

df["계절"] = df["월"].map(계절)

df[["date", "weekday", "연", "월", "주말", "계절"]].head()

## 6. 날짜를 인덱스로 — 기간 자르기

**4-6. set_index 와 기간 슬라이싱**

In [ ]:
ts = df.set_index("date")

# 문자열로 기간을 지정할 수 있다
print("2025년 3월:", len(ts.loc["2025-03"]), "일")
print("2024년 여름:", len(ts.loc["2024-06":"2024-08"]), "일")
print()
print(f"2025년 3월 평균 매출: {ts.loc['2025-03', 'sales'].mean():,.0f}원")
print(f"2024년 여름 평균 매출: {ts.loc['2024-06':'2024-08', 'sales'].mean():,.0f}원")

**4-7. resample — 기간 단위로 묶기**

In [ ]:
# ME = Month End(월말 기준). W = 주, QE = 분기, YE = 연
월별 = ts["sales"].resample("ME").mean()

print("월별 평균 매출 (만원)")
print((월별 / 10000).round(0).head(12).to_string())

## 7. 실전: 주말 효과는 얼마나 되나

**4-8. 주말 vs 평일**

In [ ]:
주말 = df[df["주말"]]
평일 = df[~df["주말"]]

print(f"주말 {len(주말):3d}일   평균 방문객 {주말['visitors'].mean():.1f}명")
print(f"평일 {len(평일):3d}일   평균 방문객 {평일['visitors'].mean():.1f}명")
print()
차이 = 주말["visitors"].mean() - 평일["visitors"].mean()
print(f"차이 {차이:+.1f}명 ({차이 / 평일['visitors'].mean() * 100:+.1f}%)")

**4-9. 퍼짐까지 함께 보기**

In [ ]:
for 이름, 그룹 in [("주말", 주말), ("평일", 평일)]:
    v = 그룹["visitors"]
    print(f"{이름}  n={len(v):3d}  평균 {v.mean():6.1f}  표준편차 {v.std():5.1f}  "
          f"범위 {v.min():.0f}~{v.max():.0f}")

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 계절별 평균 매출을 구해보세요.
#        힌트: df.groupby("계절")["sales"].mean()


# 문제 2. 2025년 데이터만 골라 월별 평균 방문객을 구해보세요.
#        힌트: df[df["연"] == 2025] 로 거르고 groupby("월")


# 문제 3. 각 달의 '일수'를 구해서 새 열로 넣어보세요.
#        힌트: df["date"].dt.days_in_month

**모범 답안**

In [ ]:
# 문제 1
계절순 = ["봄", "여름", "가을", "겨울"]
print((df.groupby("계절")["sales"].mean() / 10000).round(1).reindex(계절순))

# 문제 2
print()
print(df[df["연"] == 2025].groupby("월")["visitors"].mean().round(1))

# 문제 3
df["그달일수"] = df["date"].dt.days_in_month
print()
df[["date", "그달일수"]].head(3)

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)